In [ ]:
#@title Research methodology 2 — train, evaluate, report (about 45 min on an A100)
# Runtime > Change runtime type > A100 GPU. Outputs (checkpoints, logs, REPORT.md, for_claude.zip) go to Google Drive.
# Re-running continues where it stopped. Any setting can be overridden before `%run -i`.
import os, subprocess, sys
g = globals()
RUNS       = g.get('RUNS', 'M2:80085')    # 'M2:80085' ~45 min | 'M2:80085 M2c:80085' with PARALLEL=2 (untested speed)
PARALLEL   = g.get('PARALLEL', 1)         # trainings at once on the GPU
RESULTS    = g.get('RESULTS', '/content/drive/MyDrive/retrack_results_m2')
DATA_CACHE = g.get('DATA_CACHE', '/content/drive/MyDrive/datasets/wildtrack.tar')  # saved by attempt 1
BRANCH     = g.get('BRANCH', 'feat/research-methodology-2')
REPO_DIR   = '/content/retrack'

def sh(cmd):
    print('$', cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True, executable='/bin/bash')

from google.colab import drive
drive.mount('/content/drive')

# 1. code (already cloned by the launcher cell; cloned here when this notebook is opened on its own)
if not os.path.exists(REPO_DIR):
    sh(f'git clone -q -b {BRANCH} https://github.com/rizwanmustafa/retrack.git {REPO_DIR}')
sh(f'git -C {REPO_DIR} log --oneline -1')

# 2. WILDTRACK: this session -> Drive copy from attempt 1 -> Kaggle
W = '/content/data/Wildtrack'
have = lambda: all(os.path.exists(f'{W}/{d}') for d in ('Image_subsets', 'annotations_positions', 'calibrations'))
if not have() and DATA_CACHE and os.path.exists(DATA_CACHE):
    print('restoring WILDTRACK from Drive ...')
    os.makedirs('/content/data', exist_ok=True)
    sh(f'tar -xf {DATA_CACHE} -C /content/data')
if not have():
    print('downloading WILDTRACK from Kaggle ...')
    sh('pip -q install kagglehub')
    sh(f'cd {REPO_DIR}/WorldTrack && python -c "from retrack_pipeline import prepare_wildtrack; '
       f'prepare_wildtrack(\'/content/data\')"')
assert have(), 'WILDTRACK is not ready'

# 3. dependencies and the run (preflight ~3 min, training ~35-40 min per run, evaluation ~1 min)
sh(f'pip install -q -r {REPO_DIR}/requirements-colab.txt')
sh('nvidia-smi --query-gpu=name,memory.total --format=csv')
os.makedirs(f'{RESULTS}/m2', exist_ok=True)
sh(f'cd {REPO_DIR} && PYTHONUNBUFFERED=1 python research-methodology-2/code/m2_pipeline.py --results {RESULTS} '
   f'--runs {RUNS} --parallel {PARALLEL} 2>&1 | tee -a {RESULTS}/m2/pipeline.log; exit ${{PIPESTATUS[0]}}')

# 4. report
from IPython.display import Markdown, display
display(Markdown(open(f'{RESULTS}/m2/REPORT.md').read()))
print(f'Share {RESULTS}/m2/for_claude.zip for the analysis.')
